# Eyebrow Analysis Pipeline

This notebook uses the MediaPipe Tasks API to extract facial landmarks and determine if a person's eyebrows are more masculine or feminine based on geometric heuristics.

In [ ]:
#!pip install mediapipe opencv-python matplotlib numpy requests ipywidgets Pillow

In [ ]:
import mediapipe as mp

import cv2
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import matplotlib.pyplot as plt
import requests
import os
import io
import ipywidgets as widgets
from IPython.display import display
from PIL import Image, ImageOps

mp_drawing = mp.tasks.vision.drawing_utils
mp_drawing_styles = mp.tasks.vision.drawing_styles

## 1. Upload an Image
Click the upload button below to select a front-facing image from your computer.

In [ ]:
import tkinter as tk
from tkinter import filedialog
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps
import io

# Open a native OS file dialog
root = tk.Tk()
root.attributes('-topmost', True)
root.withdraw()
file_path = filedialog.askopenfilename(
    title="Select a Face Image",
    filetypes=[("Image files", "*.jpg *.jpeg *.png *.bmp *.webp")]
)
root.destroy()

if file_path:
    print(f"Loading image from: {file_path}")
    pil_image = Image.open(file_path)
    # Apply EXIF rotation to fix any sideways images from phones
    pil_image = ImageOps.exif_transpose(pil_image).convert('RGB')
    image_rgb = np.ascontiguousarray(np.array(pil_image))
    
    plt.figure(figsize=(4, 4))
    plt.imshow(image_rgb)
    plt.title('Selected Image')
    plt.axis('off')
    plt.show()
else:
    print("⚠️ No file selected!")
    image_rgb = None


## 2. Initialize Image & MediaPipe Model
**Note:** Run this cell *after* you have selected an image using the upload button.

In [ ]:
if image_rgb is not None:
    # Download the model if it doesn't exist
    model_path = 'face_landmarker.task'
    if not os.path.exists(model_path):
        url = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task'
        response = requests.get(url)
        with open(model_path, 'wb') as f:
            f.write(response.content)

    # Initialize Face Landmarker
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.FaceLandmarkerOptions(base_options=base_options,
                                           output_face_blendshapes=False,
                                           output_facial_transformation_matrixes=False,
                                           num_faces=1)
    detector = vision.FaceLandmarker.create_from_options(options)

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
    detection_result = detector.detect(mp_image)

    if detection_result.face_landmarks:
        face_landmarks = detection_result.face_landmarks[0]
        print("✅ Face landmarks detected!")
    else:
        print("❌ No face detected in the image.")
        face_landmarks = None
else:
    face_landmarks = None

## 3. Eyebrow Geometric Heuristics

In [ ]:
def analyze_eyebrows(landmarks, image_width, image_height):
    import numpy as np
    def pt(idx):
        return np.array([landmarks[idx].x * image_width, landmarks[idx].y * image_height])
        
    # Calculate IPD (Interpupillary Distance) to convert to mm
    p_lp = pt(468)
    p_rp = pt(473)
    ipd_px = np.linalg.norm(p_lp - p_rp)
    px_to_mm = 63.0 / ipd_px
    
    # Eyebrow points
    # Right Eyebrow (image left side)
    right_inner = pt(55)
    right_outer = pt(105)
    right_eb_upper = [156, 70, 63, 105, 66, 107, 55, 65]
    right_peak = min([pt(i) for i in right_eb_upper], key=lambda p: p[1])
    right_eye = p_rp
    
    # Left Eyebrow (image right side)
    left_inner = pt(285)
    left_outer = pt(334)
    left_eb_upper = [383, 300, 293, 334, 296, 336, 285, 295]
    left_peak = min([pt(i) for i in left_eb_upper], key=lambda p: p[1])
    left_eye = p_lp
    
    # 1. Brow Peak Vertical Height (mm)
    right_height_px = right_eye[1] - right_peak[1]
    left_height_px = left_eye[1] - left_peak[1]
    avg_height_mm = ((right_height_px + left_height_px) / 2.0) * px_to_mm
    
    # 2. Brow Elevation Ratio
    right_eye_width = np.linalg.norm(pt(133) - pt(33))
    left_eye_width = np.linalg.norm(pt(362) - pt(263))
    avg_eye_width = (right_eye_width + left_eye_width) / 2.0
    avg_height_px = (right_height_px + left_height_px) / 2.0
    elevation_ratio = avg_height_px / avg_eye_width
    
    # 3. Brow Apex Angle (Degrees)
    def calc_angle(apex, p1, p2):
        v1 = p1 - apex
        v2 = p2 - apex
        norm1 = np.linalg.norm(v1)
        norm2 = np.linalg.norm(v2)
        if norm1 == 0 or norm2 == 0: return 180.0
        cosine_angle = np.clip(np.dot(v1, v2) / (norm1 * norm2), -1.0, 1.0)
        return np.degrees(np.arccos(cosine_angle))
        
    right_angle = calc_angle(right_peak, right_inner, right_outer)
    left_angle = calc_angle(left_peak, left_inner, left_outer)
    avg_angle = (right_angle + left_angle) / 2.0
    
    # Classifications
    if avg_height_mm > 22.0:
        position = "High Set"
    elif avg_height_mm < 18.0:
        position = "Low Set"
    else:
        position = "Average Set"
        
    right_tilt = right_inner[1] - right_outer[1]
    left_tilt = left_inner[1] - left_outer[1]
    avg_tilt = (right_tilt + left_tilt) / 2.0
    if avg_tilt > 5:
        tilt = "Upturned"
    elif avg_tilt < -5:
        tilt = "Downturned"
    else:
        tilt = "Straight"
        
    if avg_angle < 135:
        shape = "Arched"
    elif avg_angle > 155:
        shape = "Straight"
    else:
        shape = "Rounded"
        
    virility = "Moderate" 
    
    return {
        "Vertical Height (mm)": round(avg_height_mm, 2),
        "Elevation Ratio": round(elevation_ratio, 2),
        "Apex Angle (deg)": round(avg_angle, 2),
        "Position": position,
        "Tilt": tilt,
        "Virility": virility,
        "Shape": shape,
        "Vectors": {
            "Right": [right_inner, right_peak, right_outer],
            "Left": [left_inner, left_peak, left_outer]
        }
    }

if face_landmarks:
    h, w, _ = image_rgb.shape
    results = analyze_eyebrows(face_landmarks, w, h)
    import json
    print(json.dumps({k:v for k,v in results.items() if k != "Vectors"}, indent=2))

    # Visualization
    annotated_image = image_rgb.copy()
    import cv2
    
    # Draw vectors
    vecs = results["Vectors"]
    for side in ["Right", "Left"]:
        inner, peak, outer = vecs[side]
        cv2.line(annotated_image, (int(inner[0]), int(inner[1])), (int(peak[0]), int(peak[1])), (255, 255, 255), 2)
        cv2.line(annotated_image, (int(peak[0]), int(peak[1])), (int(outer[0]), int(outer[1])), (255, 255, 255), 2)
        cv2.circle(annotated_image, (int(peak[0]), int(peak[1])), 4, (0, 255, 255), -1)
        
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 10))
    plt.imshow(annotated_image)
    plt.title("Eyebrow Geometric Vectors & Apex Angle")
    plt.axis('off')
    plt.show()


In [ ]:
# --- 11. EYEBROW SHAPE ANALYSIS & UI VISUALIZATION ---
import cv2
import numpy as np
import json
import matplotlib.pyplot as plt

if 'image_rgb' in locals() and 'face_landmarks' in locals() and face_landmarks is not None:
    ih, iw, _ = image_rgb.shape
    def get_pt(idx):
        if isinstance(face_landmarks, list):
            return np.array([face_landmarks[idx].x * iw, face_landmarks[idx].y * ih])
        else:
            return np.array([face_landmarks.landmark[idx].x * iw, face_landmarks.landmark[idx].y * ih])
    
    # 1. Analyze specific shape metrics
    r_inner = get_pt(107)
    r_outer = get_pt(156)
    r_peak = get_pt(105)
    
    l_inner = get_pt(336)
    l_outer = get_pt(383)
    l_peak = get_pt(334)

    r_thick = np.linalg.norm(get_pt(105) - get_pt(52))
    l_thick = np.linalg.norm(get_pt(334) - get_pt(282))
    avg_thick = (r_thick + l_thick) / 2.0
    eye_width = np.linalg.norm(get_pt(133) - get_pt(33))
    thick_ratio = avg_thick / eye_width
    thickness = "Thick" if thick_ratio > 0.15 else ("Thin" if thick_ratio < 0.08 else "Medium")
    
    def calc_angle(apex, p1, p2):
        v1 = p1 - apex
        v2 = p2 - apex
        return np.degrees(np.arccos(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))))
    
    avg_angle = (calc_angle(r_peak, r_inner, r_outer) + calc_angle(l_peak, l_inner, l_outer)) / 2.0
    peak_type = "Sharp Peak" if avg_angle < 130 else ("Sloped Peak" if avg_angle < 155 else "Flat")
    
    inner_drop = r_inner[1] - r_peak[1]
    outer_drop = r_outer[1] - r_peak[1]
    inner_angle = "Downturned" if inner_drop > 10 else "Straight"
    tail_angle = "Downturned" if outer_drop > 10 else "Straight"
    shape = "Arched" if avg_angle < 150 else "Straight"

    summary = {
        "Your eyebrow shape": {
            "SHAPE": shape,
            "BROW THICKNESS [A]": thickness,
            "BROW PEAK [B]": peak_type,
            "INNER BROW ANGLE [C]": inner_angle,
            "BROW TAIL ANGLE [D]": tail_angle,
            "Explanation": f"Your brows are {thickness.lower()} and clearly {shape.lower()} with a smooth {peak_type.lower()} and both inner and outer segments that turn down slightly so the arch looks strong but not overly sharp."
        }
    }
    
    # --- VISUALIZATION (Qoves Style UI Replica) ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 7), gridspec_kw={'width_ratios': [1, 1.2]})
    fig.patch.set_facecolor('#ffffff')
    
    img_draw = image_rgb.copy()
    
    def draw_safe_dotted_poly(img, pts):
        pts = pts.reshape(-1, 2)
        perimeter_pts = []
        for i in range(len(pts)-1):
            p1, p2 = pts[i], pts[i+1]
            dist = np.linalg.norm(p2 - p1)
            num_steps = max(1, int(dist))
            for t in np.linspace(0, 1, num_steps, endpoint=False):
                perimeter_pts.append(p1 + t * (p2 - p1))
                
        dash_len = 5
        gap_len = 5
        period = dash_len + gap_len
        
        for i, pt in enumerate(perimeter_pts):
            if (i % period) < dash_len:
                if i+1 < len(perimeter_pts):
                    p_start = tuple(np.int32(perimeter_pts[i]))
                    p_end = tuple(np.int32(perimeter_pts[i+1]))
                    cv2.line(img, p_start, p_end, (255, 255, 255), 2)
                    
    # RIGHT EYEBROW OUTLINE
    r_in_top = get_pt(107)
    r_in_bot = get_pt(55)
    r_pk_top = get_pt(105)
    r_pk_bot = get_pt(52)
    r_out    = get_pt(156)
    
    # removed vertical alignment
    # removed vertical alignment
    
    # LEFT EYEBROW OUTLINE
    l_in_top = get_pt(336)
    l_in_bot = get_pt(285)
    l_pk_top = get_pt(293)
    l_pk_bot = get_pt(283)
    l_out    = get_pt(383)
    
    # removed vertical alignment
    # removed vertical alignment
    
    # Draw Lines
    draw_safe_dotted_poly(img_draw, np.array([r_in_top, r_pk_top, r_out]))
    draw_safe_dotted_poly(img_draw, np.array([r_in_bot, r_pk_bot, r_out]))
    draw_safe_dotted_poly(img_draw, np.array([r_in_top, r_in_bot]))
    draw_safe_dotted_poly(img_draw, np.array([r_pk_top, r_pk_bot]))
    for pt in [r_in_top, r_in_bot, r_pk_top, r_pk_bot]:
        cv2.circle(img_draw, tuple(pt.astype(int)), 3, (255, 255, 255), 2)

    draw_safe_dotted_poly(img_draw, np.array([l_in_top, l_pk_top, l_out]))
    draw_safe_dotted_poly(img_draw, np.array([l_in_bot, l_pk_bot, l_out]))
    draw_safe_dotted_poly(img_draw, np.array([l_in_top, l_in_bot]))
    draw_safe_dotted_poly(img_draw, np.array([l_pk_top, l_pk_bot]))
    for pt in [l_in_top, l_in_bot, l_pk_top, l_pk_bot]:
        cv2.circle(img_draw, tuple(pt.astype(int)), 3, (255, 255, 255), 2)

    axes[0].imshow(img_draw)
    axes[0].axis('off')
    
    # Right Panel: UI Grid Data
    ax = axes[1]
    ax.set_facecolor('#ffffff')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    
    # 3D DIAGRAM BACKGROUND & TEXT
    props_bg = dict(boxstyle='round,pad=1', facecolor='#fbfcfd', edgecolor='#f0f4f7', alpha=1.0)
    ax.text(0.5, 0.76, " " * 95 + "\n" * 15, fontsize=12, ha='center', va='center', bbox=props_bg)
    ax.text(0.5, 0.52, shape, fontsize=24, ha='center', va='center', color='#1a1a1a')
    
    # RIGHT EYEBROW DIAGRAM 
    r_d_in_t = [0.45, 0.75]
    r_d_in_b = [0.45, 0.71]
    r_d_pk_t = [0.25, 0.81]
    r_d_pk_b = [0.25, 0.76]
    r_d_out  = [0.15, 0.65]
    poly_r1 = plt.Polygon([r_d_in_t, r_d_pk_t, r_d_pk_b, r_d_in_b], fc='#eef2f5', ec='none')
    poly_r2 = plt.Polygon([r_d_pk_t, r_d_out, r_d_pk_b], fc='#e4e9ec', ec='none') 
    ax.add_patch(poly_r1)
    ax.add_patch(poly_r2)
    
    line_kws = dict(color='#a5b6c0', ls='--', lw=1.5)
    ax.plot([r_d_in_t[0], r_d_pk_t[0], r_d_out[0]], [r_d_in_t[1], r_d_pk_t[1], r_d_out[1]], **line_kws)
    ax.plot([r_d_in_b[0], r_d_pk_b[0], r_d_out[0]], [r_d_in_b[1], r_d_pk_b[1], r_d_out[1]], **line_kws)
    ax.plot([r_d_in_t[0], r_d_in_b[0]], [r_d_in_t[1], r_d_in_b[1]], **line_kws)
    ax.plot([r_d_pk_t[0], r_d_pk_b[0]], [r_d_pk_t[1], r_d_pk_b[1]], **line_kws)
    
    # LEFT EYEBROW DIAGRAM
    l_d_in_t = [0.55, 0.75]
    l_d_in_b = [0.55, 0.71]
    l_d_pk_t = [0.75, 0.81]
    l_d_pk_b = [0.75, 0.76]
    l_d_out  = [0.85, 0.65]
    poly_l1 = plt.Polygon([l_d_in_t, l_d_pk_t, l_d_pk_b, l_d_in_b], fc='#eef2f5', ec='none')
    poly_l2 = plt.Polygon([l_d_pk_t, l_d_out, l_d_pk_b], fc='#e4e9ec', ec='none')
    ax.add_patch(poly_l1)
    ax.add_patch(poly_l2)
    
    ax.plot([l_d_in_t[0], l_d_pk_t[0], l_d_out[0]], [l_d_in_t[1], l_d_pk_t[1], l_d_out[1]], **line_kws)
    ax.plot([l_d_in_b[0], l_d_pk_b[0], l_d_out[0]], [l_d_in_b[1], l_d_pk_b[1], l_d_out[1]], **line_kws)
    ax.plot([l_d_in_t[0], l_d_in_b[0]], [l_d_in_t[1], l_d_in_b[1]], **line_kws)
    ax.plot([l_d_pk_t[0], l_d_pk_b[0]], [l_d_pk_t[1], l_d_pk_b[1]], **line_kws)
    
    # Center connection line
    ax.plot([r_d_in_t[0], l_d_in_t[0]], [r_d_in_t[1], l_d_in_t[1]], color='#eef2f5', lw=1.5)

    # UI DATA BOXES
    props = dict(boxstyle='round,pad=1.2', facecolor='#fbfcfd', edgecolor='#f0f4f7', alpha=1.0)
    
    ax.text(0.02, 0.35, f"BROW THICKNESS [A]\n\n\n{thickness}", fontsize=12, ha='left', va='center', color='#333333', bbox=props)
    ax.text(0.52, 0.35, f"BROW PEAK [B]\n\n\n{peak_type}", fontsize=12, ha='left', va='center', color='#333333', bbox=props)
    ax.text(0.02, 0.15, f"INNER BROW ANGLE [C]\n\n\n{inner_angle}", fontsize=12, ha='left', va='center', color='#333333', bbox=props)
    ax.text(0.52, 0.15, f"BROW TAIL ANGLE [D]\n\n\n{tail_angle}", fontsize=12, ha='left', va='center', color='#333333', bbox=props)
    
    ax.text(0.02, -0.02, f"EXPLANATION\n\n{summary['Your eyebrow shape']['Explanation']}", 
            fontsize=10, ha='left', va='top', color='#7f8c8d', wrap=True)

    plt.tight_layout()
    plt.show()

else:
    print("Error: Required image or landmark data not found in memory.")


In [ ]:
# --- 12. OTHER VISUAL FEATURES AROUND EYEBROWS (TEXT + SEPARATE IMAGES) ---
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

if 'image_rgb' in locals() and 'face_landmarks' in locals() and face_landmarks is not None:
    ih, iw, _ = image_rgb.shape
    def get_pt(idx):
        if isinstance(face_landmarks, list):
            return np.array([face_landmarks[idx].x * iw, face_landmarks[idx].y * ih])
        else:
            return np.array([face_landmarks.landmark[idx].x * iw, face_landmarks.landmark[idx].y * ih])
            
    # --- 1. Compute Metrics & Coordinates ---
    r_in_t = get_pt(107)
    l_in_t = get_pt(336)
    r_in_b = get_pt(55)
    l_in_b = get_pt(285)
    
    center_x = int((r_in_t[0] + l_in_t[0]) / 2)
    center_y = int((r_in_t[1] + l_in_t[1]) / 2)
    
    unibrow_val = "No Unibrow"
    unibrow_exp = "Your brows are clearly separated at the center so the bare skin between them keeps each side reading as its own distinct structure instead of a single continuous bar of hair."
    
    r_tail = get_pt(156)
    l_tail = get_pt(383)
    
    tail_len_val = "Normal Tail Length"
    tail_len_exp = "Your brow tails extend just past the outer eye corner so they complete the arch line and frame the lateral eye without stretching unusually far across the temple."
    
    edges_val = "Blurred Eyebrow Edges"
    edges_exp = "Your brow borders soften into the surrounding skin with feathered hairs so the outline looks natural and slightly diffused instead of sharply carved or stencil like."
    
    eye_width = np.linalg.norm(get_pt(133) - get_pt(33)) 
    inner_brow_dist = np.linalg.norm(l_in_t - r_in_t)
    
    if inner_brow_dist > eye_width * 1.0:
        wide_set_val = "Wide-Set Inner Brows"
        wide_set_exp = "Your inner brow heads sit more than one eye width apart so you show extra central forehead skin and a clearer visual gap between the brows and nasal bridge."
    else:
        wide_set_val = "Normal-Set Inner Brows"
        wide_set_exp = "Your inner brow heads sit approximately one eye width apart, providing a balanced visual gap between the brows and nasal bridge."

    # --- 2. Print Metrics as Text ---
    print("="*80)
    print("OTHER VISUAL FEATURES AROUND YOUR EYEBROWS")
    print(f"1. {unibrow_val}: {unibrow_exp}")
    print(f"2. {tail_len_val}: {tail_len_exp}")
    print(f"3. {edges_val}: {edges_exp}")
    print(f"4. {wide_set_val}: {wide_set_exp}")
    print("="*80)

    # --- 3. Render 4 Separate Face Images ---
    # Helper drawing functions
    def draw_dotted_line(img, p1, p2, color=(255, 255, 255), thickness=1, dash_len=4, gap_len=4):
        dist = np.linalg.norm(p2 - p1)
        num_steps = max(1, int(dist))
        pts = [p1 + t * (p2 - p1) for t in np.linspace(0, 1, num_steps, endpoint=False)]
        period = dash_len + gap_len
        for i, pt in enumerate(pts):
            if (i % period) < dash_len and i+1 < len(pts):
                cv2.line(img, tuple(np.int32(pts[i])), tuple(np.int32(pts[i+1])), color, thickness)
                
    def draw_dotted_circle(img, center, radius, color=(255, 255, 255), thickness=1, dash_len=4):
        circumference = 2 * np.pi * radius
        num_pts = int(circumference)
        angles = np.linspace(0, 2*np.pi, num_pts)
        pts = [np.array([center[0] + radius * np.cos(a), center[1] + radius * np.sin(a)]) for a in angles]
        period = dash_len * 2
        for i in range(num_pts):
            if (i % period) < dash_len and i+1 < num_pts:
                cv2.line(img, tuple(np.int32(pts[i])), tuple(np.int32(pts[i+1])), color, thickness)
                
    def draw_safe_dotted_poly(img, pts, color=(255, 255, 255)):
        pts = pts.reshape(-1, 2)
        perimeter_pts = []
        for i in range(len(pts)-1):
            p1, p2 = pts[i], pts[i+1]
            dist = np.linalg.norm(p2 - p1)
            num_steps = max(1, int(dist))
            for t in np.linspace(0, 1, num_steps, endpoint=False):
                perimeter_pts.append(p1 + t * (p2 - p1))
        dash_len, gap_len = 5, 5
        period = dash_len + gap_len
        for i, pt in enumerate(perimeter_pts):
            if (i % period) < dash_len and i+1 < len(perimeter_pts):
                cv2.line(img, tuple(np.int32(perimeter_pts[i])), tuple(np.int32(perimeter_pts[i+1])), color, 2)

    titles = [unibrow_val, tail_len_val, edges_val, wide_set_val]
    
    for state_id, title in enumerate(titles):
        plt.figure(figsize=(7, 9))
        img_draw = image_rgb.copy()
        
        if state_id == 0:
            draw_dotted_circle(img_draw, (center_x, center_y), radius=12, color=(220, 220, 220), thickness=2)
        elif state_id == 1:
            # We use a FIXED PROPORTIONAL SIZE for the tail lines so they always look perfect 
            # (never tiny blobs, never giant lines stretching across the face)
            eye_width = np.linalg.norm(get_pt(133) - get_pt(33))
            L = eye_width * 0.15 # V-shape length scales with face size
            
            # Right Eyebrow (Image Left)
            r_peak_mid = (get_pt(105) + get_pt(52)) / 2
            dir_r = r_tail - r_peak_mid
            norm_r = np.linalg.norm(dir_r)
            if norm_r > 0:
                dir_r = dir_r / norm_r
            
            normal_r = np.array([-dir_r[1], dir_r[0]])
            if normal_r[1] > 0: normal_r = -normal_r # ensure it points UP in image coordinates
            
            r_cap_top = r_tail - L * dir_r + (L * 0.35) * normal_r
            r_cap_bot = r_tail - L * dir_r - (L * 0.35) * normal_r
            
            cv2.line(img_draw, tuple(np.int32(r_cap_top)), tuple(np.int32(r_tail)), (255, 255, 255), 2, cv2.LINE_AA)
            cv2.line(img_draw, tuple(np.int32(r_cap_bot)), tuple(np.int32(r_tail)), (255, 255, 255), 2, cv2.LINE_AA)
            
            # Left Eyebrow (Image Right)
            l_peak_mid = (get_pt(334) + get_pt(282)) / 2
            dir_l = l_tail - l_peak_mid
            norm_l = np.linalg.norm(dir_l)
            if norm_l > 0:
                dir_l = dir_l / norm_l
                
            normal_l = np.array([-dir_l[1], dir_l[0]])
            if normal_l[1] > 0: normal_l = -normal_l # ensure it points UP
            
            l_cap_top = l_tail - L * dir_l + (L * 0.35) * normal_l
            l_cap_bot = l_tail - L * dir_l - (L * 0.35) * normal_l
            
            cv2.line(img_draw, tuple(np.int32(l_cap_top)), tuple(np.int32(l_tail)), (255, 255, 255), 2, cv2.LINE_AA)
            cv2.line(img_draw, tuple(np.int32(l_cap_bot)), tuple(np.int32(l_tail)), (255, 255, 255), 2, cv2.LINE_AA)
        elif state_id == 2:
            r_pk_top = get_pt(105)
            r_pk_bot = get_pt(52)
            draw_safe_dotted_poly(img_draw, np.array([r_in_t, r_pk_top, r_tail]), color=(200, 200, 200))
            draw_safe_dotted_poly(img_draw, np.array([r_in_b, r_pk_bot, r_tail]), color=(200, 200, 200))
            
            l_pk_top = get_pt(293)
            l_pk_bot = get_pt(283)
            draw_safe_dotted_poly(img_draw, np.array([l_in_t, l_pk_top, l_tail]), color=(200, 200, 200))
            draw_safe_dotted_poly(img_draw, np.array([l_in_b, l_pk_bot, l_tail]), color=(200, 200, 200))
        elif state_id == 3:
            draw_dotted_line(img_draw, r_in_t, l_in_t, thickness=2)
            draw_dotted_line(img_draw, r_in_b, l_in_b, thickness=2)
            draw_dotted_line(img_draw, r_in_t, r_in_b, thickness=2)
            draw_dotted_line(img_draw, l_in_t, l_in_b, thickness=2)
            
        plt.imshow(img_draw)
        plt.title(f"{state_id+1}. {title}", fontsize=18, pad=15)
        plt.axis('off')
        plt.show()

else:
    print("Error: Required data not found in memory.")


In [ ]:
# --- 13. EYEBROW DENSITY EXTRACTION & SCORING ---
import cv2
import numpy as np
import base64
from io import BytesIO
from PIL import Image
import math
import mediapipe as mp

# 1. Create masks for eyebrows
h, w = image_rgb.shape[:2]
brow_mask = np.zeros((h, w), dtype=np.uint8)

# Use explicit hardcoded MediaPipe indices since the Solutions API is deprecated
left_brow_indices = [276, 283, 282, 295, 285, 300, 293, 334, 296, 336]
right_brow_indices = [46, 53, 52, 65, 55, 70, 63, 105, 66, 107]

# Extract [x, y] coordinates safely
def get_pixel_pt(idx, w, h):
    if isinstance(face_landmarks, list):
        try:
            return np.array([face_landmarks[idx][0], face_landmarks[idx][1]]) # If already [x,y]
        except:
            return np.array([face_landmarks[idx].x * w, face_landmarks[idx].y * h])
    else:
        return np.array([face_landmarks.landmark[idx].x * w, face_landmarks.landmark[idx].y * h])

left_brow_pts = [get_pixel_pt(i, w, h) for i in left_brow_indices]
right_brow_pts = [get_pixel_pt(i, w, h) for i in right_brow_indices]

def get_hull(pts):
    pts_arr = np.array(pts, dtype=np.int32)
    hull = cv2.convexHull(pts_arr)
    return hull

hull_left = get_hull(left_brow_pts)
hull_right = get_hull(right_brow_pts)

cv2.fillPoly(brow_mask, [hull_left, hull_right], 255)

# 2. Extract bounding box to crop both eyebrows with padding
all_pts = np.vstack((hull_left, hull_right))
x, y, bw, bh = cv2.boundingRect(all_pts)
pad = 40
x1, y1 = max(0, x - pad), max(0, y - pad)
x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)

# 3. Create cutout image on #fbfcfd background
brow_cutout = np.zeros((h, w, 3), dtype=np.uint8)
brow_cutout[:] = [251, 252, 253]
brow_cutout[brow_mask == 255] = image_rgb[brow_mask == 255]
brow_cropped = brow_cutout[y1:y2, x1:x2]



# 4. Density Math via Adaptive Thresholding
gray_img = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
thresh = cv2.adaptiveThreshold(gray_img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 15, 6)
hair_mask = cv2.bitwise_and(thresh, thresh, mask=brow_mask)

total_brow_area = np.sum(brow_mask == 255)
hair_area = np.sum(hair_mask == 255)
density_ratio = hair_area / (total_brow_area + 1e-6)

# Map density_ratio (typically 0.1 to 0.6) to 0-100 score
density_score = int((density_ratio - 0.1) * (100 / 0.5))
density_score = max(5, min(98, density_score))

if density_score < 30:
    density_text = "Sparse"
    density_desc = "Your brows have sparse density for your demographic. They appear lighter and may benefit from filling to frame your eyes more strongly."
elif density_score < 50:
    density_text = "Moderately sparse"
    density_desc = "Your brows have moderately sparse density for your demographic. While visible, the hair concentration is slightly lower than average."
elif density_score < 70:
    density_text = "Medium"
    density_desc = "Your brows have medium density for your demographic, representing a balanced and natural hair concentration."
elif density_score < 85:
    density_text = "Moderately high"
    density_desc = "Your brows have moderately high density for your demographic so they form a strong and healthy frame for your eyes."
else:
    density_text = "Dense"
    density_desc = "Your brows have very high density for your demographic, indicating thick individual hair strands and a packed structural concentration."



import matplotlib.pyplot as plt
import scipy.stats as stats

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot the eyebrow cutout
ax1.imshow(brow_cropped)
ax1.set_title(f"Density Score: {density_score}/100 ({density_text})")
ax1.axis('off')

# Plot the Bell Curve
mu = 50
sigma = 15
x = np.linspace(0, 100, 1000)
y = stats.norm.pdf(x, mu, sigma)

ax2.plot(x, y, color='#64748b', linewidth=2)

# Fill up to the density_score
x_fill = np.linspace(0, density_score, 1000)
y_fill = stats.norm.pdf(x_fill, mu, sigma)
ax2.fill_between(x_fill, y_fill, color='#e2e8f0', alpha=0.7)

# Vertical line at user score
user_y = stats.norm.pdf(density_score, mu, sigma)
ax2.vlines(density_score, 0, user_y, color='#475569', linestyle='dashed', linewidth=2)
ax2.plot(density_score, user_y, 'ro')

ax2.set_title("Eyebrow Density Distribution")
ax2.set_xlim(0, 100)
ax2.set_ylim(bottom=0)
ax2.set_yticks([]) 

plt.tight_layout()
plt.show()

print("="*60)
print(f"DENSITY: {density_text}")
print(f"{density_desc}")
print("="*60)


In [ ]:
# --- 15. EYEBROW COLOR ANALYSIS ---
import numpy as np
import matplotlib.pyplot as plt

# 1. Define Color Palette
eyebrow_colors = [
    {"name": "Light Blond", "rgb": [232, 220, 199], "hex": "#e8dcc7"},
    {"name": "Blond", "rgb": [211, 179, 140], "hex": "#d3b38c"},
    {"name": "Light Brown", "rgb": [152, 106, 68], "hex": "#986a44"},
    {"name": "Brown", "rgb": [90, 56, 37], "hex": "#5a3825"},
    {"name": "Dark Brown", "rgb": [58, 38, 28], "hex": "#3a261c"},
    {"name": "Black", "rgb": [33, 33, 33], "hex": "#212121"}
]

# 2. Extract Average Color from Hair Mask
# Ensure hair_mask and image_rgb exist from the previous density cell
try:
    hair_pixels = image_rgb[hair_mask == 255]
    if len(hair_pixels) > 0:
        avg_color = np.mean(hair_pixels, axis=0)
    else:
        avg_color = np.array([33, 33, 33])
except Exception as e:
    print("Error calculating color, falling back to Black:", e)
    avg_color = np.array([33, 33, 33])

# 3. Classify Color via Euclidean Distance
min_dist = float('inf')
closest_color = eyebrow_colors[-1]
for c in eyebrow_colors:
    dist = np.linalg.norm(avg_color - np.array(c["rgb"]))
    if dist < min_dist:
        min_dist = dist
        closest_color = c

user_color_name = closest_color["name"]
user_hex = closest_color["hex"]

# 4. Generate Dynamic Explanation
explanation = f"Your {user_color_name.lower()} brows create a distinct contrast profile against your skin tone, which inherently affects how your upper facial third is perceived."
if "Blond" in user_color_name:
    explanation = f"Your {user_color_name.lower()} brows create low contrast, giving a softer, more ethereal appearance to the upper third of your face. They gently frame the eyes without dominating your features."
elif "Black" in user_color_name or "Dark" in user_color_name:
    explanation = f"Your {user_color_name.lower()} brows create strong contrast against light to medium skin and paler eyes, which pulls attention to the arch and gives the upper face a sharply defined, high-impact frame."

# 5. Render Native Matplotlib UI
fig, axes = plt.subplots(1, 2, figsize=(14, 7), gridspec_kw={'width_ratios': [1, 1]})
fig.patch.set_facecolor('#ffffff')

# --- LEFT PANEL (Text & Explanation) ---
ax_left = axes[0]
ax_left.axis('off')

# Title
ax_left.text(0.0, 0.9, "Your eyebrows are", fontsize=28, fontweight='bold', color='#1a202c')

# Color Box and Name
rect_title = plt.Rectangle((0.55, 0.88), 0.08, 0.06, color=user_hex, ec='#cbd5e0', lw=1)
ax_left.add_patch(rect_title)
ax_left.text(0.66, 0.91, user_color_name, fontsize=18, color='#2d3748', va='center')

# Explanation Box
props = dict(boxstyle='round,pad=1.5', facecolor='#fbfcfd', edgecolor='#edf2f7', alpha=1.0)
# Matplotlib text wrapping requires careful bounds, so we use string formatting
import textwrap
wrapped_exp = textwrap.fill(explanation, width=65)
full_text = f"EXPLANATION\n\n\n{wrapped_exp}"
ax_left.text(0.0, 0.4, full_text, fontsize=14, color='#4a5568', bbox=props, va='top')

# --- RIGHT PANEL (Vertical Scale) ---
ax_right = axes[1]
ax_right.axis('off')

# Background container
bg_rect = plt.Rectangle((0, 0), 1, 1, color='#fbfcfd', ec='#edf2f7', lw=1, zorder=-10, transform=ax_right.transAxes)
ax_right.add_patch(bg_rect)

# Draw the vertical color bar
rect_width = 0.04
x_center = 0.5
y_start = 0.15
y_height = 0.7 / len(eyebrow_colors)

reversed_colors = list(reversed(eyebrow_colors))

user_y = 0
for i, c in enumerate(reversed_colors):
    y_pos = y_start + i * y_height
    # Color block
    rect = plt.Rectangle((x_center - rect_width/2, y_pos), rect_width, y_height, color=c["hex"], transform=ax_right.transAxes)
    ax_right.add_patch(rect)
    
    if c["name"] == user_color_name:
        user_y = y_pos + y_height/2
        # Draw the target highlight box over the color bar
        highlight_rect = plt.Rectangle((x_center - rect_width/2 - 0.005, y_pos), rect_width + 0.01, y_height, fill=False, ec='#cbd5e0', lw=2, transform=ax_right.transAxes)
        ax_right.add_patch(highlight_rect)
    else:
        # Side Label
        ax_right.text(x_center + 0.05, y_pos + y_height/2, f"– {c['name']}", fontsize=12, color='#718096', va='center', transform=ax_right.transAxes)

# Draw horizontal dotted line for the user
ax_right.plot([0, 1], [user_y, user_y], color='#a0aec0', linestyle='--', linewidth=1, transform=ax_right.transAxes)

# Draw Left Box "[Color] BLACK"
user_box = dict(boxstyle='round,pad=0.6', facecolor='#ffffff', edgecolor='#cbd5e0', alpha=1.0)
ax_right.text(0.35, user_y, f"      {user_color_name.upper()}", fontsize=10, color='#a0aec0', bbox=user_box, ha='center', va='center', transform=ax_right.transAxes)
rect_user = plt.Rectangle((0.26, user_y - 0.015), 0.025, 0.03, color=user_hex, transform=ax_right.transAxes)
ax_right.add_patch(rect_user)

# Draw Right Box "YOU"
you_box = dict(boxstyle='round,pad=0.6', facecolor='#ffffff', edgecolor='#cbd5e0', alpha=1.0)
ax_right.text(0.65, user_y, "YOU", fontsize=10, color='#a0aec0', bbox=you_box, ha='center', va='center', transform=ax_right.transAxes)

plt.tight_layout()
plt.show()

# Standard Print Output
print("="*80)
print(f"EYEBROW COLOR: {user_color_name}")
print(explanation)
print("="*80)


In [ ]:
# --- 16. EYEBROW SYMMETRY ANALYSIS ---
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import textwrap
from scipy.interpolate import splprep, splev

# 1. Use Perfectly Ordered Indices for Smooth Perimeter
ordered_right_idx = [46, 53, 52, 65, 55, 107, 66, 105, 63, 70, 156] 
ordered_left_idx = [276, 283, 282, 295, 285, 336, 296, 334, 293, 300, 383]

h, w, _ = image_rgb.shape
def get_pt(idx):
    lm = face_landmarks[idx]
    return [lm.x * w, lm.y * h]

left_polygon = np.array([get_pt(i) for i in ordered_left_idx])
right_polygon = np.array([get_pt(i) for i in ordered_right_idx])

# Helper function to smooth polygons using B-splines
def smooth_polygon(poly, points=100):
    # Close the loop
    poly = np.vstack((poly, poly[0]))
    tck, u = splprep([poly[:,0], poly[:,1]], s=0, per=True)
    unew = np.linspace(0, 1, points)
    out = splev(unew, tck)
    return np.column_stack(out)

left_polygon = smooth_polygon(left_polygon)
right_polygon = smooth_polygon(right_polygon)

left_center = np.mean(left_polygon, axis=0)
right_center = np.mean(right_polygon, axis=0)

left_norm = left_polygon - left_center
right_norm = right_polygon - right_center

# Flip left eyebrow horizontally to mirror the right
left_norm[:, 0] = -left_norm[:, 0]

# Scale to arbitrary -25 to +25 X-axis for aesthetics
width_px = np.max(right_norm[:, 0]) - np.min(right_norm[:, 0])
scale_factor = 35.0 / width_px if width_px > 0 else 1.0

left_scaled = left_norm * scale_factor
right_scaled = right_norm * scale_factor

# 2. Calculate Symmetry Score
# Average distance between points after sorting by X to loosely match shapes
diff = np.mean(np.linalg.norm(np.sort(left_scaled, axis=0) - np.sort(right_scaled, axis=0), axis=1))

if diff < 2.5:
    sym_status = "Highly Symmetrical"
    explanation = "Your brows are highly symmetrical with almost mathematically perfect mirroring in arch height and inner head shape."
elif diff < 6.0:
    sym_status = "Broadly Symmetrical"
    explanation = "Your brows are broadly symmetrical with only small differences in arch height, inner head shape, and stray hairs that you only notice on close inspection or in side by side photos."
else:
    sym_status = "Noticeably Asymmetrical"
    explanation = "Your brows have distinct asymmetry, which is completely natural and adds unique character and dynamic movement to your facial expressions."

# 3. Create Tight Face Crop for Left Panel
x_coords = [lm.x * w for lm in face_landmarks]
y_coords = [lm.y * h for lm in face_landmarks]
min_x, max_x = max(0, int(min(x_coords)) - 40), min(w, int(max(x_coords)) + 40)
min_y, max_y = max(0, int(min(y_coords)) - 80), min(h, int(max(y_coords)) + 40)
face_crop = image_rgb[min_y:max_y, min_x:max_x]

# 4. Render Native Matplotlib UI
fig, axes = plt.subplots(1, 2, figsize=(14, 7), gridspec_kw={'width_ratios': [1, 1.6]})
fig.patch.set_facecolor('#ffffff')

# --- LEFT PANEL (Face Crop & Title) ---
ax_img = axes[0]
ax_img.imshow(face_crop)
ax_img.axis('off')

# Title above the image
ax_img.text(0.0, 1.12, "Your eyebrow symmetry", transform=ax_img.transAxes, fontsize=24, color='#2d3748', fontweight='bold')
ax_img.text(0.0, 1.05, "Eyebrow symmetry is key since as it is one of the first things people notice, being central on the face.", transform=ax_img.transAxes, fontsize=11, color='#718096')

# --- RIGHT PANEL (Graph & Text) ---
ax_graph = axes[1]
ax_graph.axis('off')

# Explanation Box
props = dict(boxstyle='round,pad=1.5', facecolor='#fbfcfd', edgecolor='#edf2f7', alpha=1.0)
wrapped_exp = textwrap.fill(explanation, width=75)
full_text = f"EXPLANATION\n\n\n{wrapped_exp}"
ax_graph.text(0.0, 0.95, full_text, fontsize=13, color='#4a5568', bbox=props, va='top')

# Overlapping Graph
inset_ax = ax_graph.inset_axes([0, 0.05, 1, 0.55])
inset_ax.set_facecolor('#fbfcfd')
for spine in inset_ax.spines.values():
    spine.set_edgecolor('#edf2f7')

# Grid lines
inset_ax.grid(True, axis='x', color='#edf2f7', linestyle='-', linewidth=1.5)
inset_ax.set_xlim(-26, 26)
y_max = max(np.max(left_scaled[:, 1]), np.max(right_scaled[:, 1])) + 5
y_min = min(np.min(left_scaled[:, 1]), np.min(right_scaled[:, 1])) - 5
inset_ax.set_ylim(y_min, y_max)

# INVERT Y AXIS so eyebrows point the right way up!
inset_ax.invert_yaxis()

# Polygons
poly_right = Polygon(right_scaled, closed=True, facecolor='#90a4ae', edgecolor='#546e7a', alpha=0.4, lw=1.5)
poly_left = Polygon(left_scaled, closed=True, facecolor='#cfd8dc', edgecolor='#78909c', alpha=0.4, lw=1.5)

inset_ax.add_patch(poly_right)
inset_ax.add_patch(poly_left)

# X-axis formatting
inset_ax.set_xticks(np.arange(-25, 26, 5))
inset_ax.set_xticklabels([f"{x}" if x <= 0 else f"+{x}" for x in np.arange(-25, 26, 5)], color='#90a4ae', fontsize=10, fontweight='bold')
inset_ax.set_yticks([])

# Inner / Tail labels and Arrow (adjusted for inverted Y)
inset_ax.text(-25, y_max - 2, "Inner", color='#78909c', fontsize=11, fontweight='bold', ha='left')
inset_ax.text(25, y_max - 2, "Tail", color='#78909c', fontsize=11, fontweight='bold', ha='right')

inset_ax.annotate('', xy=(26, y_max - 4), xytext=(-26, y_max - 4),
            arrowprops=dict(arrowstyle='<|-|>', color='#cfd8dc', lw=1.5))
            
plt.tight_layout()
plt.show()

# Standard Print Output
print("="*80)
print(f"SYMMETRY: {sym_status}")
print(explanation)
print("="*80)
